In [1]:
# =================================================================================================
# NHANES 2007-2008
# Sleep Disorders Questionnaire
#
# Source:
#   SLQ_E.xpt
#
# Purpose:
#   Convert NHANES XPT -> fidelity-preserving corrected CSV.
#
# Fidelity rules:
#
#   - Fix SEQN showing ".0"
#   - Restore ONLY validated SAS XPORT tiny-zero artifacts
#   - Preserve missing-value positions exactly
#   - Do NOT round source values
#   - Do NOT use float_format=
#   - Convert only proven whole-number/code variables to nullable Int64
#   - Validate important frequencies against official CDC documentation
#   - Verify final CSV round-trip fidelity
#
# NOTE:
#
#   SLQ_E contains NO genuine decimal variables.
#
#   The broader NHANES pipeline rule remains:
#       genuine Lab / Examination / Dietary decimals must remain AS IS.
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)


XPT_FILE = (
    DATA_DIR
    / "SLQ_E.xpt"
)


CSV_FILE = (
    DATA_DIR
    / "SLQ_E_corrected.csv"
)


# =================================================================================================
# 3. READ ORIGINAL XPT
# =================================================================================================

df = pd.read_sas(
    XPT_FILE,
    format="xport",
    encoding="latin1"
)


#
# Keep untouched pandas-decoded source for final fidelity validation.
#

source_df = df.copy()


print("=" * 110)
print("NHANES 2007-2008 SLQ_E XPT -> CSV")
print("=" * 110)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# =================================================================================================
# 4. EXPECTED OFFICIAL STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "SLD010H",
    "SLD020M",

    "SLQ030",
    "SLQ040",
    "SLQ050",
    "SLQ060",

    "SLQ070A",
    "SLQ070B",
    "SLQ070C",
    "SLQ070D",

    "SLQ080",
    "SLQ090",
    "SLQ100",
    "SLQ110",
    "SLQ120",
    "SLQ130",
    "SLQ140",
    "SLQ150",
    "SLQ160",

    "SLQ170",
    "SLQ180",
    "SLQ190",
    "SLQ200",
    "SLQ210",
    "SLQ220",
    "SLQ230",
    "SLQ240"

]


EXPECTED_ROWS = 6546


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SLQ_E columns/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != EXPECTED_ROWS:

    raise ValueError(
        f"Unexpected SLQ_E row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


print(
    "PASS: Structure = 6,546 rows x 28 columns."
)


# =================================================================================================
# 5. CONFIRM ALL FIELDS LOADED NUMERICALLY
# =================================================================================================

numeric_columns = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_columns != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 28 SLQ_E variables to load numerically."
    )


print(
    "PASS: All 28 variables loaded numerically from XPT."
)


# =================================================================================================
# 6. SAS XPORT TINY-ZERO ARTIFACT
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


ordinary_zero_counts = (
    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)


print(
    "Ordinary numeric zeros before repair:",
    f"{ordinary_zero_total:,}"
)


print(
    "Tiny XPORT values:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .to_string()
)


# =================================================================================================
# 7. VALIDATED TINY-ZERO PATTERN
# =================================================================================================

#
# Directly measured from the attached SLQ_E.xpt.
#
# CDC independently confirms zero is a legitimate value
# for these variables.
#

EXPECTED_TINY_COUNTS = {

    "SLD020M": 55,

    "SLQ030": 1936,

    "SLQ040": 4851,

    "SLQ080": 2823,

    "SLQ090": 2620,

    "SLQ100": 3057,

    "SLQ110": 2306,

    "SLQ120": 2546,

    "SLQ130": 2237,

    "SLQ140": 5419,

    "SLQ150": 5100,

    "SLQ160": 4378

}


actual_tiny_counts = (

    tiny_counts[
        tiny_counts > 0
    ]

    .astype(int)

    .to_dict()

)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: SLQ_E tiny-value pattern differs "
        "from validated attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 37328:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total:,}"
    )


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed before correction."
    )


print(
    "PASS: Exact 37,328-value tiny-zero pattern validated."
)


# =================================================================================================
# 8. RESTORE ONLY VALIDATED ZEROS
# =================================================================================================

#
# IMPORTANT:
#
# Do NOT globally replace tiny values across arbitrary NHANES files.
#
# Only these 12 SLQ_E variables have been validated.
#

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: tiny-value count changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_counts[col] = (
        actual_count
    )


total_corrections = sum(
    correction_counts.values()
)


print(
    "\n--- ZERO RESTORATION ---"
)


for col, count in correction_counts.items():

    print(
        f"{col}: {count:,}"
    )


print(
    "\nTotal restored:",
    f"{total_corrections:,}"
)


if total_corrections != 37328:

    raise ValueError(
        "Incorrect total restoration count."
    )


print(
    "PASS: 37,328 validated zeros restored."
)


# =================================================================================================
# 9. BUILD EXPECTED CORRECTED SOURCE
# =================================================================================================

expected_df = (
    source_df.copy()
)


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# =================================================================================================
# 10. VERIFY NO TINY VALUES REMAIN
# =================================================================================================

remaining_tiny = int(

    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()

)


print(
    "\nRemaining tiny XPORT values:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No XPORT tiny-zero artifacts remain."
)


# =================================================================================================
# 11. FRACTIONAL VALUE DISCOVERY
# =================================================================================================

#
# This check determines whether any variable genuinely contains
# decimal/fractional values.
#
# np.round() is used ONLY for comparison here.
# It does NOT modify the dataframe.
#

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_count = int(

            (
                ~np.isclose(
                    values,
                    np.round(values),
                    rtol=0,
                    atol=1e-12
                )
            )

            .sum()

        )


    fractional_counts[col] = (
        fractional_count
    )


    print(
        f"{col}: {fractional_count:,} fractional"
    )


fractional_variables = {

    col: count

    for col, count in fractional_counts.items()

    if count > 0

}


if fractional_variables:

    raise ValueError(
        "Unexpected genuine fractional values found:\n"
        + str(
            fractional_variables
        )
    )


print(
    "PASS: No genuine decimal variables exist in SLQ_E."
)


# =================================================================================================
# 12. FIELD TYPE PLAN
# =================================================================================================

#
# Since every nonmissing source value is whole-number after
# validated zero restoration, all 28 fields can safely be Int64.
#

INTEGER_COLUMNS = (
    EXPECTED_COLUMNS.copy()
)


DECIMAL_COLUMNS = []


print(
    "\n--- FIELD TYPE PLAN ---"
)


print(
    "Whole-number / code fields:",
    len(INTEGER_COLUMNS)
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# =================================================================================================
# 13. SEQN SOURCE VALIDATION
# =================================================================================================

print(
    "\n--- SEQN SOURCE CHECK ---"
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 41475:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 51623:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if (
    df["SEQN"]
    .isna()
    .any()
):

    raise ValueError(
        "SEQN contains missing values."
    )


if (
    df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 41475-51623 validated."
)


# =================================================================================================
# 14. CDC AGGREGATE CHECK - SLD010H
# =================================================================================================

#
# CDC:
#
# 1-11 hours       = 6498
# 12 or more hours =   40
# 99 Don't know    =    8
# Missing          =    0
#

sleep_hours_1_to_11 = int(

    df["SLD010H"]
    .between(
        1,
        11
    )
    .sum()

)


sleep_hours_12 = int(
    df["SLD010H"]
    .eq(12)
    .sum()
)


sleep_hours_99 = int(
    df["SLD010H"]
    .eq(99)
    .sum()
)


sleep_hours_missing = int(
    df["SLD010H"]
    .isna()
    .sum()
)


print(
    "\n--- CDC CHECK: SLD010H ---"
)


print(
    "1-11:",
    sleep_hours_1_to_11
)


print(
    "12:",
    sleep_hours_12
)


print(
    "99:",
    sleep_hours_99
)


print(
    "Missing:",
    sleep_hours_missing
)


if sleep_hours_1_to_11 != 6498:

    raise ValueError(
        "SLD010H 1-11 count differs from CDC."
    )


if sleep_hours_12 != 40:

    raise ValueError(
        "SLD010H=12 differs from CDC."
    )


if sleep_hours_99 != 8:

    raise ValueError(
        "SLD010H=99 differs from CDC."
    )


if sleep_hours_missing != 0:

    raise ValueError(
        "SLD010H missing count differs from CDC."
    )


print(
    "PASS: SLD010H matches CDC."
)


# =================================================================================================
# 15. CDC AGGREGATE CHECK - SLD020M
# =================================================================================================

#
# CDC:
#
# 0-59 minutes      = 5366
# 60+ coded as 60   = 1147
# 99 Don't know     =   33
# Missing           =    0
#
# Attached XPT contains exactly 55 observations at zero.
#

latency_0_to_59 = int(

    df["SLD020M"]
    .between(
        0,
        59
    )
    .sum()

)


latency_60 = int(
    df["SLD020M"]
    .eq(60)
    .sum()
)


latency_99 = int(
    df["SLD020M"]
    .eq(99)
    .sum()
)


latency_missing = int(
    df["SLD020M"]
    .isna()
    .sum()
)


print(
    "\n--- CDC CHECK: SLD020M ---"
)


print(
    "0-59:",
    latency_0_to_59
)


print(
    "60:",
    latency_60
)


print(
    "99:",
    latency_99
)


print(
    "Missing:",
    latency_missing
)


if latency_0_to_59 != 5366:

    raise ValueError(
        "SLD020M 0-59 count differs from CDC."
    )


if latency_60 != 1147:

    raise ValueError(
        "SLD020M=60 differs from CDC."
    )


if latency_99 != 33:

    raise ValueError(
        "SLD020M=99 differs from CDC."
    )


if latency_missing != 0:

    raise ValueError(
        "SLD020M missing count differs from CDC."
    )


print(
    "PASS: SLD020M matches CDC."
)


# =================================================================================================
# 16. FREQUENCY VALIDATION HELPER
# =================================================================================================

def validate_frequency(
    column,
    expected_counts,
    expected_missing=0
):

    actual_counts = (

        df[column]
        .value_counts(
            dropna=True
        )

        .sort_index()

        .to_dict()

    )


    actual_missing = int(

        df[column]
        .isna()
        .sum()

    )


    print(
        f"\n--- CDC CHECK: {column} ---"
    )


    print(
        "Observed:",
        actual_counts
    )


    print(
        "Missing:",
        actual_missing
    )


    if actual_counts != expected_counts:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n\n"
            f"Expected:\n{expected_counts}\n\n"
            f"Actual:\n{actual_counts}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# =================================================================================================
# 17. CDC FREQUENCY CHECKS
# =================================================================================================

CDC_FREQUENCIES = {

    "SLQ030": {
        0: 1936,
        1: 1016,
        2: 1124,
        3: 1882,
        7: 1,
        9: 587
    },

    "SLQ040": {
        0: 4851,
        1: 472,
        2: 388,
        3: 324,
        7: 1,
        9: 510
    },

    "SLQ050": {
        1: 1438,
        2: 5102,
        7: 1,
        9: 5
    },

    "SLQ060": {
        1: 484,
        2: 6047,
        7: 1,
        9: 14
    },

    "SLQ080": {
        0: 2823,
        1: 1224,
        2: 1359,
        3: 629,
        4: 506,
        7: 1,
        9: 4
    },

    "SLQ090": {
        0: 2620,
        1: 1179,
        2: 1482,
        3: 751,
        4: 509,
        7: 1,
        9: 4
    },

    "SLQ100": {
        0: 3057,
        1: 1130,
        2: 1238,
        3: 671,
        4: 443,
        7: 1,
        9: 6
    },

    "SLQ110": {
        0: 2306,
        1: 976,
        2: 1662,
        3: 919,
        4: 668,
        7: 1,
        9: 14
    },

    "SLQ120": {
        0: 2546,
        1: 1244,
        2: 1652,
        3: 685,
        4: 414,
        7: 1,
        9: 4
    },

    "SLQ130": {
        0: 2237,
        1: 1134,
        2: 1632,
        3: 875,
        4: 649,
        7: 1,
        9: 18
    },

    "SLQ140": {
        0: 5419,
        1: 257,
        2: 327,
        3: 169,
        4: 369,
        7: 1,
        9: 4
    },

    "SLQ150": {
        0: 5100,
        1: 482,
        2: 511,
        3: 243,
        4: 191,
        7: 1,
        9: 18
    },

    "SLQ160": {
        0: 4378,
        1: 775,
        2: 935,
        3: 287,
        4: 167,
        7: 1,
        9: 3
    },

    "SLQ170": {
        1: 82,
        2: 5045,
        3: 1118,
        4: 234,
        5: 62,
        7: 1,
        9: 4
    },

    "SLQ180": {
        1: 48,
        2: 5259,
        3: 987,
        4: 186,
        5: 57,
        7: 1,
        9: 8
    },

    "SLQ190": {
        1: 34,
        2: 6251,
        3: 201,
        4: 41,
        5: 18,
        7: 1
    },

    "SLQ200": {
        1: 290,
        2: 5470,
        3: 608,
        4: 141,
        5: 35,
        7: 1,
        9: 1
    },

    "SLQ210": {
        1: 349,
        2: 5481,
        3: 556,
        4: 122,
        5: 36,
        7: 1,
        9: 1
    },

    "SLQ220": {
        1: 394,
        2: 5441,
        3: 546,
        4: 126,
        5: 38,
        7: 1
    },

    "SLQ230": {
        1: 806,
        2: 5185,
        3: 439,
        4: 84,
        5: 31,
        7: 1
    },

    "SLQ240": {
        1: 84,
        2: 5878,
        3: 454,
        4: 89,
        5: 40,
        7: 1
    }

}


for col, expected_counts in CDC_FREQUENCIES.items():

    validate_frequency(
        col,
        expected_counts,
        expected_missing=0
    )


# =================================================================================================
# 18. CDC ROUTED SLEEP-DISORDER VARIABLES
# =================================================================================================

validate_frequency(
    "SLQ070A",
    {
        1: 280,
        9: 21
    },
    expected_missing=6245
)


validate_frequency(
    "SLQ070B",
    {
        2: 107
    },
    expected_missing=6439
)


validate_frequency(
    "SLQ070C",
    {
        3: 18
    },
    expected_missing=6528
)


validate_frequency(
    "SLQ070D",
    {
        4: 74
    },
    expected_missing=6472
)


# =================================================================================================
# 19. SLQ060 -> SLQ070 ROUTING VALIDATION
# =================================================================================================

#
# SLQ070A-D should only contain values for participants
# who answered:
#
#     SLQ060 = 1
#
# Ever told by doctor they had a sleep disorder.
#

sleep_disorder_yes = (
    df["SLQ060"]
    .eq(1)
)


print(
    "\n--- SLQ060 -> SLQ070 ROUTING ---"
)


for col in [

    "SLQ070A",
    "SLQ070B",
    "SLQ070C",
    "SLQ070D"

]:

    invalid_count = int(

        (
            df[col]
            .notna()

            &

            ~sleep_disorder_yes
        )

        .sum()

    )


    print(
        f"{col} outside SLQ060=1:",
        invalid_count
    )


    if invalid_count != 0:

        raise ValueError(
            f"{col} contains values outside "
            "SLQ060=1 routing."
        )


print(
    "PASS: SLQ070 routing validated."
)


# =================================================================================================
# 20. VERIFY CORRECTED ZERO COUNTS
# =================================================================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(

        df[col]
        .eq(0)
        .sum()

    )


    if zero_count > 0:

        final_zero_counts[
            col
        ] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final corrected zero pattern differs "
        "from validated source pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final dataframe contains exactly "
    "37,328 validated zeros."
)


# =================================================================================================
# 21. VERIFY NO OTHER SOURCE VALUE CHANGED
# =================================================================================================

pre_conversion_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_conversion_failures.append(
            col
        )


if pre_conversion_failures:

    raise ValueError(
        "Unexpected values changed during correction:\n"
        + str(
            pre_conversion_failures
        )
    )


print(
    "PASS: Only validated tiny-zero values changed."
)


# =================================================================================================
# 22. MISSING-POSITION FIDELITY BEFORE TYPE CONVERSION
# =================================================================================================

missing_position_errors = 0


for col in EXPECTED_COLUMNS:

    original_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    corrected_missing = (
        df[col]
        .isna()
        .to_numpy()
    )


    missing_position_errors += int(

        (
            original_missing
            !=
            corrected_missing
        )

        .sum()

    )


if missing_position_errors != 0:

    raise ValueError(
        "Missing-value positions changed during correction."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# =================================================================================================
# 23. CONVERT PROVEN WHOLE-NUMBER FIELDS TO NULLABLE Int64
# =================================================================================================

#
# This removes formatting such as:
#
#     SEQN = 41475.0
#
# and produces:
#
#     SEQN = 41475
#
# No genuine decimal field exists in this particular file.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        integral = np.isclose(

            values,

            np.round(
                values
            ),

            rtol=0,

            atol=1e-12

        )


        if not integral.all():

            raise ValueError(
                f"{col} contains unexpected fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All proven whole-number fields converted to Int64."
)


# =================================================================================================
# 24. PRE-EXPORT NUMERIC FIDELITY
# =================================================================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (

        pd.to_numeric(
            df[col],
            errors="coerce"
        )

        .to_numpy(
            dtype=float,
            na_value=np.nan
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export fidelity failures:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric fidelity."
)


# =================================================================================================
# 25. EXPORT CSV
# =================================================================================================

#
# IMPORTANT:
#
# No float_format=
# No rounding
#
# This is also the correct general rule for your
# Dietary/Lab/Exam files containing genuine decimals.
#

df.to_csv(

    CSV_FILE,

    index=False,

    na_rep=""

)


print(
    "\nCSV created:"
)


print(
    CSV_FILE
)


# =================================================================================================
# 26. RE-READ FINAL CSV WITH ROUND-TRIP PARSER
# =================================================================================================

roundtrip_df = pd.read_csv(

    CSV_FILE,

    low_memory=False,

    float_precision="round_trip"

)


if roundtrip_df.shape != (
    EXPECTED_ROWS,
    len(EXPECTED_COLUMNS)
):

    raise ValueError(
        "Final CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Final CSV column order changed."
    )


print(
    "PASS: Final CSV structure preserved."
)


# =================================================================================================
# 27. EXACT NUMERIC ROUND-TRIP FIDELITY
# =================================================================================================

roundtrip_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(

        expected_values,

        final_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV round-trip fidelity failures:\n"
        + str(
            roundtrip_failures
        )
    )


print(
    "PASS: All 28 variables round-trip exactly."
)


# =================================================================================================
# 28. READ FINAL CSV AS RAW TEXT
# =================================================================================================

raw_csv = pd.read_csv(

    CSV_FILE,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 29. CHECK FOR UNWANTED ".0"
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    bad = (

        raw_csv[col]
        .ne("")

        &

        raw_csv[col]
        .str.endswith(".0")

    )


    count = int(
        bad.sum()
    )


    if count > 0:

        dot_zero_errors[
            col
        ] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting remains:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' formatting."
)


# =================================================================================================
# 30. FINAL SEQN TEXT CHECK
# =================================================================================================

if (
    raw_csv["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    raw_csv["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN contains '.0'."
    )


seqn_numeric = pd.to_numeric(

    raw_csv["SEQN"],

    errors="raise"

)


if seqn_numeric.min() != 41475:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if seqn_numeric.max() != 51623:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


print(
    "PASS: Final SEQN is clean."
)


# =================================================================================================
# 31. FINAL TINY-VALUE CHECK
# =================================================================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    tiny_csv_count += int(

        values
        .eq(TINY_VALUE)
        .sum()

    )


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT values remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in final CSV."
)


# =================================================================================================
# 32. FINAL TRUE-ZERO COUNT CHECK
# =================================================================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    zero_count = int(

        values
        .eq(0)
        .sum()

    )


    if zero_count > 0:

        csv_zero_counts[
            col
        ] = zero_count


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "37,328 validated true zeros."
)


# =================================================================================================
# 33. SCIENTIFIC NOTATION CHECK
# =================================================================================================

scientific_notation_errors = {}


for col in EXPECTED_COLUMNS:

    count = int(

        raw_csv[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_errors[
            col
        ] = count


if scientific_notation_errors:

    raise ValueError(
        "Unexpected scientific notation remains:\n"
        + str(
            scientific_notation_errors
        )
    )


print(
    "PASS: No scientific notation."
)


# =================================================================================================
# 34. FINAL MISSING-POSITION CHECK
# =================================================================================================

final_missing_errors = 0


for col in EXPECTED_COLUMNS:

    original_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    final_missing = (
        raw_csv[col]
        .eq("")
        .to_numpy()
    )


    final_missing_errors += int(

        (
            original_missing
            !=
            final_missing
        )

        .sum()

    )


if final_missing_errors != 0:

    raise ValueError(
        "Final CSV missing positions differ from XPT."
    )


print(
    "PASS: Missing positions preserved through CSV export."
)


# =================================================================================================
# 35. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2007-2008 SLQ_E CONVERSION COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle: 2007-2008"
)


print(
    "Component: Questionnaire - Sleep Disorders"
)


print(
    "Eligible population: age 16+"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    ""
)


print(
    "Original XPORT tiny artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "Validated true-zero restorations:",
    f"{total_corrections:,}"
)


print(
    "Remaining tiny artifacts: 0"
)


print(
    ""
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Scientific notation: 0"
)


print(
    "Missing-position changes: 0"
)


print(
    "Round-trip numeric failures: 0"
)


print(
    "=" * 110
)

NHANES 2007-2008 SLQ_E XPT -> CSV
Rows: 6,546
Columns: 28
PASS: Structure = 6,546 rows x 28 columns.
PASS: All 28 variables loaded numerically from XPT.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before repair: 0
Tiny XPORT values: 37,328

Tiny values by variable:
SLD020M      55
SLQ030     1936
SLQ040     4851
SLQ080     2823
SLQ090     2620
SLQ100     3057
SLQ110     2306
SLQ120     2546
SLQ130     2237
SLQ140     5419
SLQ150     5100
SLQ160     4378
PASS: Exact 37,328-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
SLD020M: 55
SLQ030: 1,936
SLQ040: 4,851
SLQ080: 2,823
SLQ090: 2,620
SLQ100: 3,057
SLQ110: 2,306
SLQ120: 2,546
SLQ130: 2,237
SLQ140: 5,419
SLQ150: 5,100
SLQ160: 4,378

Total restored: 37,328
PASS: 37,328 validated zeros restored.

Remaining tiny XPORT values: 0
PASS: No XPORT tiny-zero artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional
SLD010H: 0 fractional
SLD020M: 0 fractional
SLQ030: 0 fractional
SLQ040: 0 fractional
SLQ050: 0